# Transactions — 02: Bulk Feature Ingest

**Persona:** Data Engineer

**Goal:** Submit a GeoJSON `FeatureCollection` of 50 synthetic administrative boundary
points (Horn of Africa extent) via a single bulk POST to the OGC Features Transactions
endpoint. Verify the `BulkCreationResponse`, then paginate through the results using
the `next` link token.

---

## Prerequisites

- DynaStore running and reachable at `DYNASTORE_BASE_URL`
- Catalog `CATALOG_ID` exists
- Collection `COLLECTION_ID` exists inside that catalog with a LayerConfig assigned
- `DYNASTORE_WRITE_TOKEN` set if the instance requires authentication

## Environment variables

| Variable | Default | Description |
|---|---|---|
| `DYNASTORE_BASE_URL` | `http://localhost:8080` | API base URL |
| `DYNASTORE_WRITE_TOKEN` | _(empty)_ | Bearer token for write operations |
| `CATALOG_ID` | `demo-catalog` | Target catalog |
| `COLLECTION_ID` | `adm-boundaries` | Target collection |

In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL      = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
WRITE_TOKEN   = os.environ.get("DYNASTORE_WRITE_TOKEN", "")
CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-catalog")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "adm-boundaries")

headers = {"Authorization": f"Bearer {WRITE_TOKEN}"} if WRITE_TOKEN else {}
client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60)

print(f"Base URL    : {BASE_URL}")
print(f"Catalog     : {CATALOG_ID}")
print(f"Collection  : {COLLECTION_ID}")
print(f"Auth header : {'set' if WRITE_TOKEN else 'not set'}")

## Step 1 — Generate synthetic test data

Generate 50 point features scattered across the Horn of Africa bounding box
(lon 33–42, lat 3–12). Each feature carries a stable `external_id` used as
the conflict-resolution identity key by DynaStore's write policy matcher.

In [ ]:
import random
import json

random.seed(42)  # reproducible run

features = []
for i in range(50):
    lon = random.uniform(33.0, 42.0)  # Horn of Africa bbox
    lat = random.uniform(3.0, 12.0)
    features.append({
        "type": "Feature",
        "id": f"adm_boundary_{i:03d}",
        "geometry": {
            "type": "Point",
            "coordinates": [round(lon, 6), round(lat, 6)]
        },
        "properties": {
            "external_id": f"ADM_{i:03d}",
            "name": f"Admin Unit {i}",
            "region": "HOA"
        }
    })

feature_collection = {
    "type": "FeatureCollection",
    "features": features
}

print(f"Generated {len(features)} features")
print("First feature:")
print(json.dumps(features[0], indent=2))

## Step 2 — Bulk POST

Submit the entire `FeatureCollection` in a single request. The OGC Features
Transactions endpoint accepts both a single `Feature` and a `FeatureCollection`.
For a bulk submission the server processes all features atomically inside one
database transaction and returns a `BulkCreationResponse` with a `created` count.

In [ ]:
resp = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=feature_collection,
)
assert resp.status_code == 201, (
    f"Expected 201 Created, got {resp.status_code}: {resp.text[:400]}"
)

bulk_response = resp.json()
print("BulkCreationResponse:")
print(json.dumps(bulk_response, indent=2))

In [ ]:
# Verify the server confirms it created all 50 features.
# The response key may be "created" (count) or "ids" (list) depending on platform version.
created_count = bulk_response.get("created", len(bulk_response.get("ids", [])))
assert created_count == 50, (
    f"Expected 50 features created, got {created_count}. "
    f"Full response: {json.dumps(bulk_response)}"
)
print(f"Confirmed: {created_count} features created")

## Step 3 — Paginate results

List items with `limit=10` to receive the first page. The response includes
a `next` link (OGC Features pagination). Extract the `href` from that link
and follow it to confirm the second page is reachable.

In [ ]:
resp = client.get(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"limit": 10},
)
assert resp.status_code == 200, (
    f"Expected 200 OK, got {resp.status_code}: {resp.text[:400]}"
)

page1 = resp.json()
page1_features = page1.get("features", [])
assert len(page1_features) == 10, (
    f"Expected 10 features on page 1, got {len(page1_features)}"
)

# Extract next link
next_link = next(
    (lnk["href"] for lnk in page1.get("links", []) if lnk.get("rel") == "next"),
    None,
)

print(f"Page 1: {len(page1_features)} features returned")
print(f"numberMatched: {page1.get('numberMatched', 'n/a')}")
print(f"Next link: {next_link}")
for feat in page1_features[:3]:
    print(f"  - {feat['id']} @ {feat['geometry']['coordinates']}")
print("  ...")

In [ ]:
# Follow the next link to retrieve page 2
assert next_link is not None, (
    "No 'next' link returned — collection may have fewer than 10 items or pagination is disabled"
)

# The next link may be absolute or relative; httpx handles both.
# If it is absolute we use a bare client to avoid double-prefixing the base URL.
if next_link.startswith("http"):
    resp2 = httpx.get(next_link, headers=headers, timeout=30)
else:
    resp2 = client.get(next_link)

assert resp2.status_code == 200, (
    f"Expected 200 OK on page 2, got {resp2.status_code}: {resp2.text[:400]}"
)

page2 = resp2.json()
page2_features = page2.get("features", [])
print(f"Page 2: {len(page2_features)} features returned")
for feat in page2_features[:3]:
    print(f"  - {feat['id']} @ {feat['geometry']['coordinates']}")
print("  ...")

## Edge cases

### Case A — FeatureCollection with one invalid feature

Submitting a `FeatureCollection` where one feature has a structural error
(e.g. `geometry` key missing entirely, not the same as `null`) should trigger
a validation error. DynaStore processes bulk inserts inside a single transaction,
so a validation failure on any feature causes the **entire batch to roll back**.
No partial writes occur.

In [ ]:
bad_feature = {
    "type": "Feature",
    "id": "bad_feature_no_geometry_key",
    # 'geometry' key is entirely absent — not null, just missing
    "properties": {
        "external_id": "BAD_001",
        "name": "Invalid Admin Unit",
        "region": "HOA"
    }
}

bad_collection = {
    "type": "FeatureCollection",
    "features": [
        features[0],   # valid
        bad_feature,   # invalid — missing geometry key
        features[1],   # valid
    ]
}

resp_bad = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    json=bad_collection,
)
# Expected: 422 Unprocessable Entity (Pydantic validation failure) or
#           400 Bad Request (schema rejection before DB write).
# The entire batch is rolled back — the two valid features are NOT stored.
print(f"POST with invalid feature → {resp_bad.status_code}")
if resp_bad.status_code in (400, 422):
    print("  Batch rejected as expected — atomic rollback confirmed")
    error_detail = resp_bad.json()
    print(f"  Error: {json.dumps(error_detail, indent=2)[:400]}")
else:
    print(f"  Unexpected status — review platform validation behaviour")
    print(f"  Body: {resp_bad.text[:300]}")

## Teardown

Delete all 50 ingested items. If the platform exposes a purge endpoint for the
collection it is used first; otherwise items are deleted individually.

In [ ]:
# Attempt collection-level purge first (faster, single request)
resp_purge = client.delete(
    f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
    params={"purge": "true"},
)

if resp_purge.status_code in (200, 204):
    print(f"Collection purged — status {resp_purge.status_code}")
else:
    print(
        f"Purge endpoint returned {resp_purge.status_code}; "
        "falling back to per-item deletion."
    )
    deleted = 0
    failed = []
    for feat in features:
        r = client.delete(
            f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items/{feat['id']}"
        )
        if r.status_code == 204:
            deleted += 1
        else:
            failed.append((feat["id"], r.status_code))

    print(f"Deleted {deleted}/{len(features)} items")
    if failed:
        print(f"Failed deletions ({len(failed)}): {failed[:5]}")

client.close()